In [1]:
import numpy as np
import pandas as pd

In [2]:
# Setup fake sky frame and target list (from presentation)
rng = np.random.default_rng(42)
img = rng.normal(100, 5, (64, 64))
img[10, 20] = 5000  # hot pixel artifact

cat = pd.DataFrame({
    'name': ['V883 Ori', 'BN/KL', 'Fomalhaut', 'HD 214734', 'AB Aur'],
    'ra'  : [83.05, 83.81, 344.41, 226.62, 73.94],
    'dec' : [-7.15, -5.37, -29.62, 61.30, 30.55],
    'kmag': [8.4, 6.1, 0.9, 7.2, 4.2],
    'filt': ['Ks', 'Ks', 'H', 'Ks', 'H'],
})

In [3]:
# 1 & 2. Mask hot pixels (> 3 sigma above median) and compare means
mean_before = img.mean()

med = np.median(img)
std = np.std(img)

In [5]:
# Identify and mask hot pixels
hot_pixels = img > (med + 3 * std)
img_clean = img.copy()
img_clean[hot_pixels] = np.nan

mean_after = np.nanmean(img_clean)
diff = mean_before - mean_after

print("--- Image Cleaning ---")
print(f"Mean before: {mean_before:.4f}")
print(f"Mean after : {mean_after:.4f}")
print(f"Difference : {diff:.4f}\n")

--- Image Cleaning ---
Mean before: 101.1015
Mean after : 99.9051
Difference : 1.1963



In [6]:
# 3. Filter targets (dec > -10) and group by filter
print("--- Target Filtering & Grouping ---")
dec_filtered = cat[cat['dec'] > -10]
print("Targets with dec > -10:")
print(dec_filtered[['name', 'ra', 'dec', 'kmag', 'filt']], "\n")

mean_kmag = cat.groupby('filt')['kmag'].mean()
print("Mean kmag per filter:")
print(mean_kmag)

--- Target Filtering & Grouping ---
Targets with dec > -10:
        name      ra    dec  kmag filt
0   V883 Ori   83.05  -7.15   8.4   Ks
1      BN/KL   83.81  -5.37   6.1   Ks
3  HD 214734  226.62  61.30   7.2   Ks
4     AB Aur   73.94  30.55   4.2    H 

Mean kmag per filter:
filt
H     2.550000
Ks    7.233333
Name: kmag, dtype: float64


## Task Summary

* **Image Masking:** Identified the cosmic ray / hot pixel at `[10, 20]` using a 3-sigma threshold above the median (`img > median + 3 * std`). Replacing it with `NaN` dropped the frame's mean background from ~101.1 back to the expected nominal noise level (~99.9).
* **Catalog Query:** Filtered targets with `dec > -10` and calculated the mean `kmag` per filter band (`filt`) across the target catalogue.